# Digitalização 3D do cardápio — a metade que precisa de GPUEste caderno faz **só a parte cara**: transformar a foto de cada prato numa malha 3D.Ele sobe a malha crua para o Supabase e para por aí.O acabamento — encaixar a escala em metros, tirar a mesa que o gerador reconstróijunto, simplificar e comprimir — acontece depois, na máquina de quem desenvolve, com`pnpm modelos:gerar --acabar`. Isso é de propósito: aquele código já existe, já foidepurado, e reescrevê-lo aqui em Python significaria manter duas versões da mesmaregra, que divergiriam na primeira correção feita só de um lado.## Antes de rodar1. **Ligue a GPU**: menu da direita → *Session options* → *Accelerator* → **GPU T4 x2**2. **Cadastre os segredos**: *Add-ons* → *Secrets* → adicione dois:   - `SUPABASE_URL` — o endereço do projeto (`https://....supabase.co`)   - `SUPABASE_SERVICE_KEY` — a chave `service_role`A chave `service_role` ignora todas as regras de segurança do banco. Ela fica nosSecrets do Kaggle, nunca escrita numa célula — célula fica salva no caderno, e cadernopúblico vaza a chave para qualquer um.## O que esperarA instalação leva **20 a 30 minutos** e acontece de novo a cada sessão nova, porque oKaggle apaga o disco ao fechar. Depois disso, cada prato leva de 1 a 3 minutos.A T4 tem 16 GB, que é o **mínimo** do TRELLIS a 512³. Se aparecer erro de memória,a última célula explica o que trocar.

## 1. InstalaçãoDemora. Rode e vá fazer outra coisa.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheaderimport os, subprocess, sysos.environ['ATTN_BACKEND'] = 'xformers'os.environ['SPCONV_ALGO'] = 'native'# O repositório traz submódulos com extensões CUDA que precisam ser compiladas.!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /kaggle/working/TRELLIS 2>&1 | tail -3%cd /kaggle/working/TRELLIS!pip install -q --upgrade pip!pip install -q pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless \    scipy ninja rembg onnxruntime trimesh xatlas pyvista pymeshfix igraph transformers 2>&1 | tail -3!pip install -q xformers --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3!pip install -q git+https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -3!pip install -q spconv-cu120 2>&1 | tail -3!pip install -q supabase 2>&1 | tail -2print('\n--- instalação terminada ---')

## 2. Conexão com o bancoLê os segredos e mostra o que está esperando geração.

In [ ]:
from kaggle_secrets import UserSecretsClientfrom supabase import create_clientseg = UserSecretsClient()sb = create_client(seg.get_secret('SUPABASE_URL'), seg.get_secret('SUPABASE_SERVICE_KEY'))BUCKET = 'product-models'BUCKET_FOTOS = 'product-photos'def fila():    """Pratos com foto e sem modelo pronto.    A fila é lida do banco a cada rodada, e não fixada no início: se a máquina de    casa acabar um prato enquanto este caderno roda, ele some da fila sozinho.    """    prontos = {        m['product_id']        for m in sb.table('product_models').select('product_id,status').execute().data        if m['status'] in ('pronto', 'processando')    }    produtos = sb.table('products').select('id,name,restaurant_id,image_url').order('sort_order').execute().data    return [p for p in produtos if p['image_url'] and p['id'] not in prontos]pendentes = fila()print(f'{len(pendentes)} prato(s) esperando geração:')for p in pendentes[:10]:    print('  ·', p['name'])if len(pendentes) > 10:    print(f'  … e mais {len(pendentes) - 10}')

## 3. Carregar o modeloBaixa os pesos uma vez e deixa na memória da GPU.

In [ ]:
import syssys.path.append('/kaggle/working/TRELLIS')from trellis.pipelines import TrellisImageTo3DPipelinepipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')pipe.cuda()print('modelo na GPU')

## 4. GerarUm prato por vez, gravando **assim que cada um fica pronto**. Se a sessão cair no meio— e sessão gratuita cai —, o que já foi feito está salvo e a próxima rodada continuade onde parou.

In [ ]:
import io, time, tracebackfrom PIL import Imageimport imageioLIMITE = None   # None = todos; ponha um número para testar com poucosalvos = pendentes if LIMITE is None else pendentes[:LIMITE]feitos, falhas = 0, 0for p in alvos:    inicio = time.time()    try:        foto = sb.storage.from_(BUCKET_FOTOS).download(p['image_url'])        imagem = Image.open(io.BytesIO(foto)).convert('RGB')        # `preprocess_image` remove o fundo. Vale muito num cardápio: foto de prato        # vem com toalha, talher e mão de garçom em volta.        saida = pipe.run(imagem, seed=42,                         sparse_structure_sampler_params={'steps': 12, 'cfg_strength': 7.5},                         slat_sampler_params={'steps': 12, 'cfg_strength': 3.0})        from trellis.utils import postprocessing_utils        glb = postprocessing_utils.to_glb(            saida['gaussian'][0], saida['mesh'][0],            simplify=0.95, texture_size=1024,        )        buf = io.BytesIO()        glb.export(buf, file_type='glb')        bytes_glb = buf.getvalue()        caminho = f"brutos/{p['restaurant_id']}/{p['id']}.glb"        sb.storage.from_(BUCKET).upload(            caminho, bytes_glb,            {'content-type': 'model/gltf-binary', 'upsert': 'true'},        )        # 'processando' com bruto preenchido é o sinal que a máquina de casa espera.        sb.table('product_models').upsert({            'product_id': p['id'],            'restaurant_id': p['restaurant_id'],            'status': 'processando',            'origem': 'foto',            'provedor': 'trellis-kaggle',            'bruto_path': caminho,            'segundos': round(time.time() - inicio, 1),        }, on_conflict='product_id').execute()        feitos += 1        print(f"  ✓ {p['name'][:34]:34} {len(bytes_glb)/1024:6.0f} KB  {time.time()-inicio:5.0f}s")    except Exception as e:        falhas += 1        msg = str(e)[:200]        print(f"  ✗ {p['name'][:34]:34} {msg}")        sb.table('product_models').upsert({            'product_id': p['id'], 'restaurant_id': p['restaurant_id'],            'status': 'falhou', 'origem': 'foto', 'provedor': 'trellis-kaggle',            'erro': msg,        }, on_conflict='product_id').execute()print(f'\n{feitos} gerados, {falhas} falharam.')

## 5. Terminar, na sua máquinaAs malhas estão no bucket. Para virarem cardápio, rode aí:```bashpnpm modelos:gerar --acabar```Ele baixa cada bruto, encaixa a escala em metros, remove a mesa, gera os dois níveis(leve para a lista, pesado para o AR), comprime com Draco e publica. Só CPU.---## Se der erro de memóriaA T4 de 16 GB é o mínimo do TRELLIS. Duas saídas, em ordem:**Baixar os passos.** Na célula 4, troque `'steps': 12` por `'steps': 8` nos doislugares. Menos detalhe, bem menos memória.**Trocar de modelo.** O [Hunyuan3D-2GP](https://github.com/deepbeepmeep/Hunyuan3D-2GP)é feito para pouca VRAM e roda com 6 GB. Só a célula 3 muda; o resto do cadernocontinua igual, porque o que importa daqui para frente é o GLB no bucket.## Se a sessão cairAcontece — a gratuita tem limite de tempo. Nada se perde: cada prato é gravadoassim que fica pronto, e reabrir o caderno e rodar tudo de novo pula o que jáestá feito.